# Assignment 4
## UCS420 - Cognitive Computing
### Topic: A Cognitive FAQ System Using Pandas (Nova 2.0)

**Name:** Gurliv Singh

**Roll Number:** 1024170161

**Course:** UCS420 - Cognitive Computing

**Assignment:** 4 - Cognitive FAQ System

In [ ]:
import pandas as pd

---
### Q1. Build Your Personalized Knowledge Base
Roll number: **1024170161** (last two digits: **6, 1**)
- digit 6 -> 6 % 3 = 0 -> **billing**
- digit 1 -> 1 % 3 = 1 -> **account**

In [ ]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", 
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", 
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", 
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", 
     "keywords": "pay payment upi fee", "category": "billing"},
]

# Roll number digits: 6 -> 6%3=0 -> billing, 1 -> 1%3=1 -> account
personalized_entries = [
    {"question": "what is the late payment fee", 
     "answer": "A late payment fee of Rs 100 is charged after the due date.", 
     "keywords": "late fine penalty fee", "category": "billing"},
    {"question": "how do i change my registered email", 
     "answer": "Go to Account Settings > Profile to update your registered email address.", 
     "keywords": "change update email address account", "category": "account"},
]

all_entries = fixed_entries + personalized_entries
df_faq = pd.DataFrame(all_entries)
print("FAQ Knowledge Base (6 entries):")
print(df_faq.to_string())


### Q2. Scoring Function - Match Query and Rank by Confidence

In [ ]:
def score_query(query, df):
    """Score each FAQ entry based on keyword matches with the query."""
    query_words = set(query.lower().split())
    scores = []
    for _, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        
        # Count matches with keywords + question
        kw_match = len(query_words & keywords)
        q_match = len(query_words & question_words)
        confidence = kw_match * 2 + q_match  # keywords weighted more
        scores.append(confidence)
    
    df_result = df.copy()
    df_result["confidence"] = scores
    df_result = df_result[df_result["confidence"] > 0].sort_values("confidence", ascending=False)
    return df_result

# Test
query = "how to pay fee"
print(f"Query: '{query}'")
result = score_query(query, df_faq)
print("\nMatching entries (ranked by confidence):")
print(result[["question", "answer", "confidence"]].to_string())

### Q3. same_category() Function

In [ ]:
def same_category(category_name, df):
    """Return all questions belonging to a given category."""
    return df[df["category"] == category_name]

# Call using 'billing' (one of the personalized entry categories)
print("Questions in 'billing' category:")
billing_faqs = same_category("billing", df_faq)
print(billing_faqs[["question", "answer", "category"]].to_string())

### Q4. Add a New Keyword and Save to CSV

In [ ]:
# Pick entry at index 0 (annual fee)
new_keyword = "annual"  # simulating user input
df_faq.at[0, "keywords"] = df_faq.at[0, "keywords"] + " " + new_keyword
print("Updated keywords for entry 0:", df_faq.at[0, "keywords"])

# Save to CSV with roll number
roll_number = "1024170161"
filename = f"{roll_number}_faq_data.csv"
df_faq.to_csv(filename, index=False)
print(f"\nSaved FAQ data to: {filename}")


### Q5. Count FAQ Entries Per Category Using groupby

In [ ]:
print("FAQ entries per category:")
category_count = df_faq.groupby("category").size().reset_index(name="count")
print(category_count)

### Q6. Modified Scoring with Tie Handling

In [ ]:
def score_query_with_ties(query, df):
    """Score entries and handle ties - print ALL top matches if tie occurs."""
    query_words = set(query.lower().split())
    scores = []
    for _, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        kw_match = len(query_words & keywords)
        q_match = len(query_words & question_words)
        confidence = kw_match * 2 + q_match
        scores.append(confidence)
    
    df_result = df.copy()
    df_result["confidence"] = scores
    df_result = df_result[df_result["confidence"] > 0]
    
    if df_result.empty:
        print("No matching entries found.")
        return
    
    max_score = df_result["confidence"].max()
    top_entries = df_result[df_result["confidence"] == max_score]
    
    if len(top_entries) > 1:
        print(f"TIE detected! {len(top_entries)} entries with max confidence {max_score}:")
        print(top_entries[["question", "answer", "confidence"]].to_string())
    else:
        print(f"Top match (confidence: {max_score}):")
        print(top_entries[["question", "answer", "confidence"]].to_string())

# Demo 1: Query that produces a TIE (matches both fee entries)
print("=== Query producing a TIE ===")
score_query_with_ties("fee", df_faq)

# Demo 2: Query with no tie
print("\n=== Query with no tie ===")
score_query_with_ties("reset password login", df_faq)